# ENSEMBLE METHODS AND AUTOML

In this homework you will apply ensemble learning concepts from the paired lesson using scikit-learn's ensemble tools on the Wine Recognition dataset - a 3-class classification problem with 13 chemical measurements. You will measure how model diversity affects accuracy, build soft-voting ensembles, and study how threshold selection changes the precision-recall tradeoff.

***Summary***
1. [Understanding Model Diversity](#section-diversity)
2. [Building Ensemble Classifiers](#section-ensemble)
3. [Combining Predictions and Threshold Analysis](#section-threshold)
4. [Multi-Domain Generalization](#section-generalization)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split, cross_val_predict
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

sns.set(style='white', context='notebook', palette='deep')
plt.rcParams['figure.figsize'] = 10, 5
RANDOM_STATE = 42

In [ ]:
# Load Wine dataset and split into train/test sets
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name='target')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples')
print(f'Classes: {sorted(y.unique().tolist())} | Class distribution: {y.value_counts().to_dict()}')

***
<a id='section-diversity'></a>
## 1. Understanding Model Diversity

The lesson explained that ensemble methods reduce error when constituent models make different mistakes. This section measures that effect directly: compare a single decision tree (high variance) to a random forest (bagging reduces variance) using 5-fold cross-validation.

**Q1 a) Train a `DecisionTreeClassifier` and a `RandomForestClassifier` (n_estimators=100) on the training data using 5-fold cross-validation (cv=5, scoring='accuracy'). Store the mean OOF accuracy for the decision tree in `dt_mean_acc` and for the random forest in `rf_mean_acc`. Store the standard deviations in `dt_cv_std` and `rf_cv_std`. Print all four values.**

Expected output:

<img src="assets/hw2_q1a_expected.png" width="450" />

*Hint:* Use [`sklearn.model_selection.cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html). The `.mean()` and `.std()` of the returned array give the quantities you need.

In [ ]:
n_folds = 5  # don't change this line

### YOUR CODE HERE ###
dt_scores = ...
rf_scores = ...

dt_mean_acc = ...
rf_mean_acc = ...
dt_cv_std   = ...
rf_cv_std   = ...

#print(f'Decision Tree:  mean={dt_mean_acc:.3f}, std={dt_cv_std:.3f}')
#print(f'Random Forest:  mean={rf_mean_acc:.3f}, std={rf_cv_std:.3f}')

**Q1 b) Using the standard deviations you computed in Q1a, explain in 2-3 sentences what `dt_cv_std` and `rf_cv_std` each measure in the context of bias-variance tradeoff, and why the random forest typically has lower standard deviation than a single decision tree.**

*ANSWER HERE*

***
<a id='section-ensemble'></a>
## 2. Building Ensemble Classifiers

This section builds a soft-voting ensemble from three diverse base classifiers - mirroring how the lesson combined LightAutoML, FlaML, and H2O predictions. The ensemble strategy here is sklearn's `VotingClassifier` rather than AutoML, but the underlying principle (average probability outputs from diverse models) is identical.

**Q2 a) Train three classifiers on the training data using 5-fold cross-validation (cv=5, scoring='accuracy'): (1) `RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)`, (2) `GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)`, and (3) `LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)`. Store each fitted model in `clf_rf`, `clf_gb`, `clf_lr`. Store their mean OOF accuracies in `acc_rf`, `acc_gb`, `acc_lr`.**

Expected output:

<img src="assets/hw2_q2a_expected.png" width="450" />

*Hint:* Call `cross_val_score` on each classifier, then `.fit()` on the training data to produce the fitted models you will use in Q2b and Q2c.

In [ ]:
### YOUR CODE HERE ###

# Instantiate the three classifiers
clf_rf = ...
clf_gb = ...
clf_lr = ...

# Cross-validate each and store mean OOF accuracy
acc_rf = ...
acc_gb = ...
acc_lr = ...

# Fit all three on the full training set for use in Q2b and Q2c
...

#print(f'RandomForest    OOF accuracy: {acc_rf:.3f}')
#print(f'GradientBoosting OOF accuracy: {acc_gb:.3f}')
#print(f'LogisticRegression OOF accuracy: {acc_lr:.3f}')

**Q2 b) Combine the three fitted classifiers from Q2a into a `VotingClassifier` with `voting='soft'`. Store it in `clf_voting`. Evaluate it using 5-fold cross-validation and store the mean OOF accuracy in `acc_voting`. In the markdown cell below, state whether the ensemble beats the best individual model and by how much.**

Expected output:

<img src="assets/hw2_q2b_expected.png" width="450" />

*Hint:* [`sklearn.ensemble.VotingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html) takes a list of `(name, estimator)` tuples. Soft voting averages probability outputs; hard voting uses majority class labels.

In [ ]:
### YOUR CODE HERE ###
clf_voting = ...
acc_voting = ...

#print(f'VotingClassifier OOF accuracy: {acc_voting:.3f}')
#print(f'Best individual:               {max(acc_rf, acc_gb, acc_lr):.3f}')

*ANSWER HERE*

**Q2 c) Compute the predicted probability for each class from each of the three fitted classifiers on the test set. For each classifier, take the maximum probability across classes (the confidence in the predicted class) and create a Series indexed by test sample. Compute the Pearson correlation matrix among the three classifiers' confidence scores. Store the correlation DataFrame in `pred_corr`. In the markdown cell below, identify which two classifiers are most similar and explain in one sentence why that makes sense given how they work.**

Expected output:

<img src="assets/hw2_q2c_expected.png" width="400" />

*Hint:* Each fitted classifier's `.predict_proba(X_test)` returns an (n_samples, n_classes) array. `.max(axis=1)` gives the highest probability per sample. Use [`pd.DataFrame.corr()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) to compute pairwise correlations.

In [ ]:
### YOUR CODE HERE ###
conf_rf = ...
conf_gb = ...
conf_lr = ...

pred_corr = ...

#print('Pairwise correlation of classifier confidence scores:')
#print(pred_corr.round(3))

*ANSWER HERE*

***
<a id='section-threshold'></a>
## 3. Combining Predictions and Threshold Analysis

The lesson showed that shifting the decision threshold from 0.5 changes how many survivors are predicted. This section applies the same idea to a one-vs-rest binary view of the wine dataset: wine class 1 vs. not class 1.

**Q3 a) Using the fitted `clf_rf` from Q2a, compute precision, recall, and F1 score for predicting wine class 1 (vs. all other classes) on the test set at three thresholds: 0.3, 0.5, and 0.7. Use `clf_rf.predict_proba(X_test)[:, 1]` as the probability score. Store the results in a DataFrame named `threshold_df` with columns `['threshold', 'precision', 'recall', 'f1']` and one row per threshold.**

Expected output:

<img src="assets/hw2_q3a_expected.png" width="400" />

*Hint:* Create a binary label `y_binary = (y_test == 1).astype(int)`. For each threshold, compute `y_pred = (proba >= threshold).astype(int)` and use [`precision_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html), [`recall_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html), and [`f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html).

In [ ]:
thresholds = [0.3, 0.5, 0.7]  # don't change this line
y_binary = (y_test == 1).astype(int)  # don't change this line
proba_class1 = clf_rf.predict_proba(X_test)[:, 1]  # don't change this line

### YOUR CODE HERE ###
rows = []
for th in thresholds:
    ...

threshold_df = ...

#print(threshold_df.round(3).to_string(index=False))

**Q3 b) Using the results in `threshold_df`, explain in 3-4 sentences why lowering the decision threshold increases recall but decreases precision. Then give one real-world scenario (not wine classification) where a high-recall, low-precision classifier would be preferable to a high-precision, low-recall one. Be specific about what the false positives and false negatives represent in your scenario.**

*ANSWER HERE*

***
<a id='section-generalization'></a>
## 4. Multi-Domain Generalization

This section confirms you can transfer the simple averaging ensemble strategy to a different binary classification problem. The Breast Cancer Wisconsin dataset is binary (benign vs. malignant), so AUC is a natural metric here - directly comparable to the lesson's OOF AUC comparison between LightAutoML and FlaML.

**Q4 a) Load `load_breast_cancer()`, split it 80/20 train/test with `random_state=RANDOM_STATE`, and train the same three classifiers from Q2a (RandomForest, GradientBoosting, LogisticRegression) on the training set. For each classifier, compute the OOF AUC on the training set using `cross_val_predict` with `method='predict_proba'`. Then compute the simple average of the three classifiers' probability predictions on the test set and compute the ensemble's test AUC. Store all four AUC scores in a DataFrame named `bc_results` with columns `['model', 'auc']`.**

Expected output:

<img src="assets/hw2_q4a_expected.png" width="400" />

*Hint:* [`cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html) with `method='predict_proba'` returns out-of-fold probability predictions. For a binary problem, use `[:, 1]` to get the positive-class probability. Simple average: `(proba_rf + proba_gb + proba_lr) / 3`.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_predict

bc = load_breast_cancer()
X_bc = pd.DataFrame(bc.data, columns=bc.feature_names)
y_bc = pd.Series(bc.target, name='target')

X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=RANDOM_STATE, stratify=y_bc
)  # don't change the above lines

### YOUR CODE HERE ###

# Re-instantiate classifiers for breast cancer (same params as Q2a)
bc_rf = ...
bc_gb = ...
bc_lr = ...

# OOF probability predictions on training set (use cross_val_predict)
oof_rf = ...
oof_gb = ...
oof_lr = ...

# Fit on training set for test prediction
...

# Test probability predictions
test_proba_rf = ...
test_proba_gb = ...
test_proba_lr = ...

# Simple average ensemble on test set
ensemble_proba = ...

# Compute OOF AUC for each model and test AUC for ensemble
auc_rf  = roc_auc_score(y_bc_train, oof_rf)
auc_gb  = roc_auc_score(y_bc_train, oof_gb)
auc_lr  = roc_auc_score(y_bc_train, oof_lr)
auc_ens = roc_auc_score(y_bc_test, ensemble_proba)

bc_results = pd.DataFrame({
    'model': ['RandomForest (OOF)', 'GradientBoosting (OOF)', 'LogisticRegression (OOF)', 'Ensemble (test)'],
    'auc': [auc_rf, auc_gb, auc_lr, auc_ens]
})

#print(bc_results.round(4).to_string(index=False))

**Q4 b) Based on the AUC scores in `bc_results` and the prediction correlation you computed for the wine dataset in Q2c: explain in 2-3 sentences whether you expected the ensemble to outperform the best individual model on the breast cancer dataset, and why. Reference the model diversity principle from the lesson.**

*ANSWER HERE*